<a href="https://colab.research.google.com/github/WahidSaeed/event_bliss_workshops/blob/main/bliss_pathology_foundation_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pathology Foundation Models**
---

This notebook is a hands-on introduction to **pathology foundation models**, large neural networks pre-trained on millions of histopathology images. These models learn rich visual representations of tissue that can be reused across many downstream tasks: cancer subtyping, survival prediction, biomarker detection, and more, without requiring task-specific labels for pre-training.

## 0 - Install and import dependencies

In [ ]:
# Run this cell once to make sure all required packages are available
!pip install -q transformers datasets Pillow torch torchvision requests matplotlib scikit-learn

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
from datasets import load_dataset
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from transformers import AutoImageProcessor, AutoModel

## 1 - Load an example pathology foundation model

We use [Phikon-v2](https://huggingface.co/owkin/phikon-v2) as a first example which is a Vision Transformer (ViT-L) pre-trained with DINOv2 on a corpus of TCGA whole-slide images. The model outputs a 1024-dimensional feature vector for each 224 × 224 image patch. That vector is a compact representation of everything the model has learned to see in the tissue.

In [ ]:
processor = AutoImageProcessor.from_pretrained("owkin/phikon-v2")
model = AutoModel.from_pretrained("owkin/phikon-v2")
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on {device}.")

## 2 - Dataset: Camelyon Patches from PathoRob

We will use a curated subset of the **Camelyon** dataset, which consists of histopathology patches from lymph node sections. Metastatic breast cancer occurs when cancer cells break away from the main tumor in the breast and travel through the lymphatic system to nearby lymph nodes. Detecting these metastases is a critical task for pathologists.

Each 256 × 256 H&E patch in this dataset includes:
- **biological_class**: Whether the tissue is 'normal' or 'tumor' (metastatic).
- **medical_center**: The specific hospital or facility where the sample was collected, which can introduce variations in staining and scanning.

In [ ]:
DATASET_ID = "bifold-pathomics/PathoROB-camelyon"

print(f"Loading dataset '{DATASET_ID}' ...")
dataset = load_dataset(DATASET_ID)
print(dataset)

# Use the test split for exploration (or 'train' if test is unavailable)
split_name = "train"
split = dataset[split_name]
print(f"\nUsing split: '{split_name}' — {len(split)} samples")
print(f"Columns: {split.column_names}")

label_col = 'biological_class'
NORMAL_LABEL = 'normal'
TUMOR_LABEL  = 'tumor'

normal_indices = [i for i, lbl in enumerate(split[label_col]) if lbl == NORMAL_LABEL]
tumor_indices  = [i for i, lbl in enumerate(split[label_col]) if lbl == TUMOR_LABEL]

print(f"\nNormal patches : {len(normal_indices):,}")
print(f"Tumor patches  : {len(tumor_indices):,}")

### Visualise sample patches

To get a better sense of the data, we will plot a few random samples from each category. The visualization below displays two rows of image patches:

- The **top row** shows patches labeled as **Normal** tissue.
- The **bottom row** shows patches labeled as **Tumor** tissue (metastatic breast cancer).

In [ ]:
N_SAMPLES = 6  # patches per class to display

random.seed(15)
sample_normal = random.sample(normal_indices, N_SAMPLES)
sample_tumor  = random.sample(tumor_indices,  N_SAMPLES)

image_col = next(c for c in split.column_names if c.lower() in ("image", "img", "pixel_values"))

fig, axes = plt.subplots(2, N_SAMPLES, figsize=(3 * N_SAMPLES, 7))
fig.suptitle("PathoROB-Camelyon — sample patches", fontsize=14, fontweight="bold")

for col, idx in enumerate(sample_normal):
    img = split[idx][image_col]
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    axes[0, col].imshow(img)
    axes[0, col].axis("off")
    if col == 0:
        axes[0, col].set_ylabel("Normal", fontsize=12, rotation=0, labelpad=55, va="center")

for col, idx in enumerate(sample_tumor):
    img = split[idx][image_col]
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    axes[1, col].imshow(img)
    axes[1, col].axis("off")
    if col == 0:
        axes[1, col].set_ylabel("Tumor", fontsize=12, rotation=0, labelpad=55, va="center")

plt.tight_layout()
plt.show()

## 3 - Extract embeddings

We now run a small subset of patches through the foundation model to get their 1024-d feature vectors. This is the core operation in any foundation-model pipeline: the model transforms raw pixel data into a semantically meaningful representation.

In [ ]:
from tqdm.auto import tqdm

N_EMBED = 200  # patches per class

random.seed(0)
embed_normal = random.sample(normal_indices, min(N_EMBED, len(normal_indices)))
embed_tumor  = random.sample(tumor_indices,  min(N_EMBED, len(tumor_indices)))
all_indices  = embed_normal + embed_tumor
all_labels   = [0] * len(embed_normal) + [1] * len(embed_tumor)


def embed_patches(indices, forward_fn, batch_size=32):
    """Return a (N, D) tensor of embeddings.

    forward_fn: list[PIL.Image] -> (B, D) tensor
    """
    vectors = []
    for start in tqdm(range(0, len(indices), batch_size)):
        batch_idx = indices[start : start + batch_size]
        imgs = [
            split[idx][image_col] if isinstance(split[idx][image_col], Image.Image)
            else Image.fromarray(split[idx][image_col])
            for idx in batch_idx
        ]
        with torch.inference_mode():
            vecs = forward_fn(imgs)
        vectors.append(vecs.cpu())
    return torch.cat(vectors, dim=0)


def phikon_forward(imgs):
    inp = processor(imgs, return_tensors="pt").to(device)
    out = model(**inp)
    return out.last_hidden_state[:, 0, :]  # CLS token, (B, D)


print("Extracting embeddings — this may take a minute on CPU ...")
embeddings = embed_patches(all_indices, phikon_forward)
labels     = torch.tensor(all_labels)
print(f"Done. Embeddings shape: {embeddings.shape}")
np.savez("phikon_embeddings.npz", embeddings=embeddings.numpy())
print("Saved to phikon_embeddings.npz")

## 4 - Visualise the embedding space with UMAP

1024 dimensions is hard to look at. UMAP projects them down to 2-D while preserving neighbourhood structure. If the foundation model has learned meaningful tissue representations, tumor and normal patches should form distinct clusters

In [ ]:
# install UMAP for dimensionality reduction and visualization
!pip install -q umap-learn

In [ ]:
import umap
import numpy as np


def compute_umap(embeddings):
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    return reducer.fit_transform(embeddings.numpy())


def plot_by_class(coords, labels, title):
    fig, ax = plt.subplots(figsize=(8, 6))
    for label_val, color, name in [(0, "#4C72B0", "Normal"), (1, "#DD8452", "Tumor")]:
        mask = labels == label_val
        ax.scatter(coords[mask, 0], coords[mask, 1], c=color, label=name, alpha=0.7, s=40, edgecolors="none")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.legend(title="Class", framealpha=0.9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


phikon_coords = compute_umap(embeddings)
plot_by_class(phikon_coords, labels, "Phikon-v2 embeddings — UMAP projection")

## 5 - Visualise the embedding space by Medical Center

Looking at the UMAP projection above, the model has clearly learned to separate tumor from normal tissue: the two classes form distinct regions. But look more carefully: there are **two large clusters**, and within each cluster both tumor and normal patches are present. The class boundary runs *within* each cluster, not *between* them.

This means something other than biology is driving the top-level split. The model has picked up on a second source of variation in the data that is strong enough to pull patches apart before it even gets to tumor vs. normal.

A natural suspect is **technical confounders**: differences in staining protocols, scanners, or tissue preparation across hospitals. These differences affect the raw pixel statistics of every patch, regardless of whether it contains cancer cells.

To test this hypothesis, we colour the same UMAP projection by **medical center**. If the two large clusters align with different institutions, that is a strong signal that the embedding space is capturing site-level batch effects alongside the biological signal.

In [ ]:
def plot_by_medical_center(coords, medical_center_labels, title):
    unique_centers = sorted(set(medical_center_labels))
    cmap = plt.get_cmap('tab10' if len(unique_centers) <= 10 else 'viridis', len(unique_centers))
    center_colors_map = {c: cmap(i) for i, c in enumerate(unique_centers)}
    fig, ax = plt.subplots(figsize=(8, 6))
    for center_name in unique_centers:
        mask = [lbl == center_name for lbl in medical_center_labels]
        ax.scatter(coords[mask, 0], coords[mask, 1], c=[center_colors_map[center_name]], label=center_name, alpha=0.7, s=40, edgecolors="none")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.legend(title="Medical Center", framealpha=0.9, bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


medical_center_labels = [split[idx]['medical_center'] for idx in all_indices]
plot_by_medical_center(phikon_coords, medical_center_labels, "Phikon-v2 embeddings — UMAP projection by Medical Center")

## 6 - Visualize Sample Patches for UMCU and RUMC

The medical center coloring makes the pattern unmistakable: the two large clusters correspond almost perfectly to the two institutions. This confirms our suspicion: the top-level structure in the embedding space is driven by **technical confounders**, not biology.

Before drawing further conclusions, it is worth asking whether these differences are actually visible to the human eye, or whether the model is picking up on subtle low-level statistics that are imperceptible to us. We therefore sample a few patches from each medical center and display them side by side.

In [ ]:
N_SAMPLES_PER_CENTER = 6 # Number of patches to display per medical center

# Filter indices for UMCU and RUMC
umcu_indices = [idx for i, idx in enumerate(all_indices) if medical_center_labels[i] == 'UMCU']
rumc_indices = [idx for i, idx in enumerate(all_indices) if medical_center_labels[i] == 'RUMC']

random.seed(42) # For reproducibility
sample_umcu = random.sample(umcu_indices, min(N_SAMPLES_PER_CENTER, len(umcu_indices)))
sample_rumc = random.sample(rumc_indices, min(N_SAMPLES_PER_CENTER, len(rumc_indices)))

fig, axes = plt.subplots(2, N_SAMPLES_PER_CENTER, figsize=(3 * N_SAMPLES_PER_CENTER, 7))
fig.suptitle("Sample Patches from UMCU and RUMC Medical Centers", fontsize=14, fontweight="bold")

# Display UMCU patches
for col, idx in enumerate(sample_umcu):
    img = split[idx][image_col]
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    axes[0, col].imshow(img)
    axes[0, col].axis("off")
    axes[0, col].set_title("UMCU") # Add title to each UMCU patch

# Display RUMC patches
for col, idx in enumerate(sample_rumc):
    img = split[idx][image_col]
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    axes[1, col].imshow(img)
    axes[1, col].axis("off")
    axes[1, col].set_title("RUMC") # Add title to each RUMC patch

plt.tight_layout()
plt.show()

## 7 - Downstream Implications: Does the Site Matter?

Looking at the patches side by side, the visual differences are immediately apparent and concentrated in **color**: RUMC patches have a more intense purple/violet hue, while UMCU patches lean toward a softer pink. These are staining differences introduced during slide preparation, entirely unrelated to whether the tissue contains cancer.

**Why does it matter?**

If the embedding space encodes site-specific staining, a classifier can exploit it as a shortcut. To expose this, we construct a training set with a strong spurious correlation:

- **90% of tumor patches** come from UMCU, 10% from RUMC
- **90% of normal patches** come from RUMC, 10% from UMCU

The biased pool is split 50/50 into train and val. The test set is all remaining patches not selected into train or val. A linear classifier that has learned "UMCU = tumor, RUMC = normal" should perform well on val (same bias) but fail on the unbiased test set.

In [ ]:
def train_probe(embeddings, labels, mask):
    X, y = embeddings.numpy(), labels.numpy()
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X[mask], y[mask])
    return clf

def eval_probe(clf, embeddings, labels, mask, name):
    X, y = embeddings.numpy()[mask], labels.numpy()[mask]
    acc = accuracy_score(y, clf.predict(X))
    print(f"  {name}: {acc:.2%}  ({mask.sum()} samples)")

labels_np     = labels.numpy()
mc_labels_arr = np.array(medical_center_labels)

tumor_center_probs  = {'UMCU': 0.9, 'RUMC': 0.1}
normal_center_probs = {'UMCU': 0.1, 'RUMC': 0.9}

rng = np.random.default_rng(42)

def build_biased_pool(label, center_probs, rng):
    idx_sets = [np.where((labels_np == label) & (mc_labels_arr == c))[0] for c in center_probs]
    probs    = list(center_probs.values())
    n        = int(min(len(idx) / p for idx, p in zip(idx_sets, probs)))
    pool     = np.concatenate([rng.choice(idx, round(n * p), replace=False) for idx, p in zip(idx_sets, probs)])
    return rng.permutation(pool)

def split_pool(pool, train_frac=0.5):
    n = int(train_frac * len(pool))
    return pool[:n], pool[n:]

train_tumor,  val_tumor  = split_pool(build_biased_pool(1, tumor_center_probs,  rng))
train_normal, val_normal = split_pool(build_biased_pool(0, normal_center_probs, rng))

train_mask = np.zeros(len(all_labels), dtype=bool)
val_mask   = np.zeros(len(all_labels), dtype=bool)
train_mask[np.concatenate([train_tumor, train_normal])] = True
val_mask[np.concatenate([val_tumor,   val_normal])]     = True
test_mask = ~(train_mask | val_mask)

for name, mask in [("Train", train_mask), ("Val", val_mask), ("Test", test_mask)]:
    print(f"{name}: {mask.sum():3d}  ({(labels_np[mask]==0).sum()} normal / {(labels_np[mask]==1).sum()} tumor)")

clf = train_probe(embeddings, labels, train_mask)
print()
eval_probe(clf, embeddings, labels, val_mask,  "Val  (biased, same distribution as train)")
eval_probe(clf, embeddings, labels, test_mask, "Test (all remaining patches)")

## 8 - RudolfV2: A More Robust Foundation Model

RudolfV2 is a pathology foundation model developed by us at [Aignostics](https://www.aignostics.com), planned for open-source release in the coming weeks with similiar robustness improvements as Atlas 2. You can sign up [under this link](https://docs.google.com/forms/d/e/1FAIpQLSdksVEeyL1IzjU1FUrCNGsNugOIQFB9bZj869e8uSJGUHYinQ/viewform) to get notified when it becomes available.

We embed the same patches using RudolfV2. Pre-computed embeddings can be loaded from Google Drive as it is not on Huggingface yet.

In [ ]:
# Option A: extract embeddings directly using the model (requires access to model weights)
import timm
from aimm.models.vision_transformer import DinoVisionTransformer

rudolfv2 = timm.create_model(
    'aignx_vit',
    cfg='/app/rudolfv2/config.yaml',
    pretrained=True,
    pretrained_cfg_overlay={"file": "/app/rudolfv2/teacher_checkpoint.pth"}
)
rudolfv2 = rudolfv2.to(device)
rudolfv2.eval()

data_cfg = timm.data.resolve_data_config(rudolfv2.pretrained_cfg)
transform = timm.data.create_transform(**data_cfg)

def rudolfv2_forward(imgs):
    batch = torch.stack([transform(img) for img in imgs]).to(device)
    return rudolfv2(batch)

print("Extracting RudolfV2 embeddings ...")
rudolf_embeddings = embed_patches(all_indices, rudolfv2_forward)
print(f"Done. Embeddings shape: {rudolf_embeddings.shape}")
np.savez("rudolfv2_embeddings.npz", embeddings=rudolf_embeddings.numpy())
print("Saved to rudolfv2_embeddings.npz")

In [ ]:
# Option B: load pre-computed embeddings from Google Drive
!pip install -q gdown

import gdown
import os
import numpy as np

RUDOLF_GDRIVE_URL = "https://drive.google.com/uc?id=1EQs83-ZLThq-8PZZyX7u7taf0R4QuOPC"
RUDOLF_EMBEDDINGS_PATH = "rudolfv2_embeddings.npz"

if not os.path.exists(RUDOLF_EMBEDDINGS_PATH):
    print("Downloading pre-computed RudolfV2 embeddings ...")
    gdown.download(RUDOLF_GDRIVE_URL, RUDOLF_EMBEDDINGS_PATH, quiet=False)

data = np.load(RUDOLF_EMBEDDINGS_PATH)
rudolf_embeddings = torch.tensor(data["embeddings"], dtype=torch.float32)
print(f"Loaded RudolfV2 embeddings: {rudolf_embeddings.shape}")

In [ ]:
rudolf_coords = compute_umap(rudolf_embeddings)
plot_by_class(rudolf_coords, labels, "RudolfV2 — UMAP projection")
plot_by_medical_center(rudolf_coords, medical_center_labels, "RudolfV2 — UMAP projection by Medical Center")

### The embedding space looks much better

The RudolfV2 UMAP projection looks different. Tumor and normal patches separate more cleanly into distinct regions and the tight medical center clusters that dominated the Phikon-v2 projection are far less pronounced. Patches from different institutions are now much more interleaved, suggesting that the model is encoding tissue morphology rather than site-specific artefacts.

**Does this robustness translate to downstream classification?**

We run the same spurious correlation experiment from Section 7: train a linear probe on the biased split (90% of tumor from UMCU, 90% of normal from RUMC) and evaluate on the unbiased test set. If the embeddings no longer carry strong center information, the classifier has less of a shortcut to exploit.

In [ ]:
rudolf_clf = train_probe(rudolf_embeddings, labels, train_mask)
print("RudolfV2 linear probe (same biased split as Section 7):")
eval_probe(rudolf_clf, rudolf_embeddings, labels, val_mask,  "Val  (biased, same distribution as train)")
eval_probe(rudolf_clf, rudolf_embeddings, labels, test_mask, "Test (all remaining patches)")

### The Bias in the Representation Propagates Downstream

With Phikon-v2, the biased training set allowed the classifier to exploit center-specific staining as a shortcut, producing a noticeable accuracy drop when evaluated on the unbiased test set. With RudolfV2, val and test performance remain close. The classifier has little shortcut to exploit because the embeddings no longer encode strong site-level signals.